## Retrieval-Augmented Generation (RAG) AND Evaluation
### Using PDF and TXT files as the knowledge source

### **Retrieval-Augmented Generation (RAG)**

Retrieval-Augmented Generation (RAG) is a powerful approach that combines information retrieval with text generation. Instead of relying solely on a language model's internal knowledge, RAG retrieves relevant documents from an external source before generating responses. This enhances accuracy, reduces hallucinations, and provides up-to-date information.

In this lesson, we will implement RAG using the **OpenAI API** for both **document embeddings** and **text generation**.

We use `ChatOpenAI` and `OpenAIEmbeddings` throughout. Because Together AI also
exposes an **OpenAI-compatible endpoint**, the exact same code runs against
Together simply by changing the API key and the base URL. You will see how in
the setup cell below.

---

### Setting Up the Environment

Before we begin, ensure you have the necessary libraries installed:

In [19]:
%pip install langchain langchain-community langchain-openai chromadb pypdf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


**Import the required libraries:**

- `PyPDFLoader`: Reads a PDF file and returns one document per page.
- `RecursiveCharacterTextSplitter`: Cuts long documents into smaller chunks.
- `Document`: The LangChain container for a piece of text.
- `OpenAIEmbeddings`: Embeds text into vector representations.
- `Chroma`: Stores and retrieves embedded documents efficiently.
- `ChatOpenAI`: Uses an AI model for text generation.
- `ChatPromptTemplate` and `StrOutputParser`: Create structured prompts and extract responses.
- `dotenv` :  This helps to load the environment variables like API Keys.

In [26]:
pip install langchain_chroma

  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)
Using cached langchain_chroma-1.1.0-py3-none-any.whl (12 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
from langchain_chroma import Chroma
import os
import chromadb
import pathlib
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
#from langchain.schema import Document
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

**Loading the Documents**

For this tutorial, we will use two separate files as our knowledge sources:

- `msme_policy.pdf` - a PDF about the National Policy on MSMEs in Nigeria.
- `business_registration_guide.txt` - a text file about registering a business with the CAC.

These are two **different** sources. We load them separately and keep them
separate, so you can decide which one to put in the vector database.

**How we read each file**

- The PDF needs a special reader, because a PDF is not plain text. We use
  `PyPDFLoader`, which returns **one document per page**.
- The text file needs nothing special. We open it with plain Python
  `with open(...)`, exactly the way you learned in the file handling topic.

`with open(...)` is the normal way to read a file in Python. The `with` block
closes the file automatically when it finishes, even if something goes wrong.

*You can swap in your own PDF or text file later. Just change the two file names below.*

In [29]:
# Source 1 - the PDF. A PDF is not plain text, so we need a PDF reader.
# This gives us one document per page.
pdf_docs = PyPDFLoader("msme_policy.pdf").load()

# Source 2 - the text file. Plain Python can read this on its own.
with open("business_registration_guide.txt", "r", encoding="utf-8") as f:
    txt_text = f.read()

print("Pages loaded from the PDF :", len(pdf_docs))
print("Characters read from TXT  :", len(txt_text))

print()
print("First 300 characters of the PDF:")
print(pdf_docs[0].page_content[:300])

print()
print("First 300 characters of the TXT:")
print(txt_text[:300])

Pages loaded from the PDF : 4
Characters read from TXT  : 4601

First 300 characters of the PDF:
NATIONAL POLICY ON MICRO, SMALL AND MEDIUM ENTERPRISES (MSMEs)
Federal Republic of Nigeria
1. INTRODUCTION
Micro, Small and Medium Enterprises (MSMEs) are the backbone of the Nigerian
economy. They account for the overwhelming majority of registered businesses in
the country, contribute close to hal

First 300 characters of the TXT:
GUIDE TO REGISTERING A BUSINESS IN NIGERIA
Corporate Affairs Commission (CAC)

WHY REGISTER

Registration with the Corporate Affairs Commission gives a business a legal
identity. Without it a business cannot open a corporate bank account, bid for
government or corporate contracts, obtain most catego


**Splitting the documents into chunks**

A whole PDF page or a whole text file is usually too long to embed as a single
vector. Long text also makes retrieval less precise, because one vector has to
represent many different ideas at once.

So we cut each document into smaller overlapping **chunks**:

- `chunk_size=800` - each chunk is about 800 characters.
- `chunk_overlap=100` - each chunk repeats the last 100 characters of the one
  before it, so a sentence is not cut in half and loses its meaning.

We split the PDF and the text file **separately**, so they stay as two distinct
sets of chunks.

The splitter has two methods, and we need one of each here:

- `split_documents(...)` - use this when you already have documents, like the
  pages that came back from `PyPDFLoader`.
- `create_documents([...])` - use this when you have **plain text strings**, like
  the text we just read with `with open(...)`. It cuts the text up and wraps each
  piece into a document for us.

After this step both sources look the same, so the rest of the notebook does not
care which file a chunk came from.

In [30]:
# The same splitter is used for both sources
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

# The PDF pages are already documents, so we split the documents
pdf_chunks = splitter.split_documents(pdf_docs)

# The TXT is one long plain string, so we create documents from it
txt_chunks = splitter.create_documents([txt_text])

print("Chunks from the PDF :", len(pdf_chunks))
print("Chunks from the TXT :", len(txt_chunks))


# ---------------------------------------------------------------
# CHOOSE WHICH SOURCE TO EMBED
# Uncomment the line you want and comment out the others.
# ---------------------------------------------------------------
chosen_chunks = pdf_chunks + txt_chunks     # use both together
# chosen_chunks = pdf_chunks                # use only the PDF
# chosen_chunks = txt_chunks                # use only the text file


# Chroma only needs the plain text of each chunk
documents = [chunk.page_content for chunk in chosen_chunks]

print()
print("Documents ready to embed:", len(documents))

Chunks from the PDF : 12
Chunks from the TXT : 7

Documents ready to embed: 19


`documents` is now a simple list of text strings. That is all we need to embed.

**Which source answers which question?**

The two files cover different ground, so the source you pick decides which
questions the model can answer:

| Question asked later in this notebook | Answered by |
|---|---|
| How long does a business name reservation last? | the TXT file |
| What is the first step in setting up a business? | the TXT file |
| How do I register a construction business? | the TXT file |
| The five RAGAS evaluation questions (MSME classification, SMEDAN/BoI/CBN, sectors, challenges, interventions) | the PDF |

That is why the default above is **both together**. If you switch to only one
source, expect the questions belonging to the other file to return nothing
useful - which is itself a good thing to show students about RAG.

**If you change source, reset the vector store.** Either change
`collection_name` in the Chroma cell below, or delete the `chroma_store` folder.
Otherwise the new text is mixed in with whatever you embedded before.

In [31]:
#use this to comfirm the number documents to embed 
print(f"Total Documents to be embedded {len(documents)}\n")

Total Documents to be embedded 19



This loads the environment variables from your `.env` file, which contains the API key.

**Choosing a provider**

We use `ChatOpenAI` and `OpenAIEmbeddings` for everything. Together AI supports
the OpenAI-compatible endpoint, so switching provider means changing only the
API key, the base URL and the model names - never the code that uses them.

Uncomment the block you want below.

In [34]:
load_dotenv()

# ---------------------------------------------------------------
# CHOOSE YOUR PROVIDER
# Uncomment one block. The rest of the notebook does not change.
# ---------------------------------------------------------------

# --- Option 1: OpenAI ---
api_key = os.environ["OPENAI_API_KEY"]
base_url = "https://api.openai.com/v1"
chat_model_name = "gpt-4o-mini"
embedding_model_name = "text-embedding-3-small"

# --- Option 2: Together AI (OpenAI-compatible endpoint) ---
# api_key = os.environ["TOGETHER_API_KEY"]
# base_url = "https://api.together.xyz/v1"
# chat_model_name = "meta-llama/Llama-3.3-70B-Instruct-Turbo"
# embedding_model_name = "togethercomputer/m2-bert-80M-32k-retrieval"

print("Using base URL      :", base_url)
print("Chat model          :", chat_model_name)
print("Embedding model     :", embedding_model_name)

KeyError: 'OPENAI_API_KEY'

In [33]:
LOCAL_API_BASE = "https://qwen-embed.publicaai.com/v1"
LOCAL_API_KEY = "publica"
model="Qwen/Qwen3-Embedding-0.6B"

**Setting Up ChromaDB for embeddings Storage**

chroma_dir: Gets the current working directory where the notebook is located.

chroma_path: Creates a folder named chroma_store to store the vector embeddings.

In [36]:
chroma_path = "chroma_store"


- **We are using the `embedding_model_name` you chose above** to convert documents into vector embeddings.
- For OpenAI that is `text-embedding-3-small`; for Together it is `togethercomputer/m2-bert-80M-32k-retrieval`.

**Important:** different embedding models produce vectors of different sizes
(1536 numbers for `text-embedding-3-small`, 768 for the Together m2-bert model).
Chroma cannot mix the two in one collection. If you change the embedding model,
delete the `chroma_store` folder or use a new `collection_name`, or you will get
a dimension mismatch error.

In [38]:
# Initialize the embedding model
embeddings = OpenAIEmbeddings(
    model=model,
    api_key="publica",
    base_url="https://qwen-embed.publicaai.com/v1"

    # Only needed when you point this at Together instead of OpenAI.
    # OpenAIEmbeddings counts tokens using OpenAI's own tokenizer, which does
    # not know non-OpenAI model names. Turning the check off stops that error.
    # check_embedding_ctx_length=False
)

- Creates a **vector database (Chroma)** for the documents.
- Adds the **embedded texts** to the database.

In [41]:
# Initialize ChromaDB
msmevdb = Chroma(
    collection_name="msme",
    embedding_function=embeddings,
    persist_directory=chroma_path
)


In [42]:
# Add documents to the vector database
msmevdb.add_texts(
    texts=documents
)

['c3a285c6-dde8-464c-a70a-bfb44c904adb',
 'f26552a1-c810-4249-a616-d119bf3b0475',
 'adca2d13-f612-4bdd-b58b-5e5c7183e7ce',
 '6bd056af-18cd-416a-ae1d-9bc78c6ea1bb',
 '728223e4-89cb-4590-9d47-c42bd4b0bd6c',
 '55b2649a-995a-4b83-9f05-d38096943bc2',
 '1ba492c2-6c91-46e1-8863-e2e01361a557',
 '4fc4ae8b-393c-4200-8388-d1c2d6990742',
 '9dbe7c6d-0eb2-4f87-bdce-ab9da7697c0e',
 '5055a5cf-ac56-453d-a502-7cb61c9b3402',
 'dfd1d0e2-65f2-44c9-8ca0-8c25bbb1e3b6',
 '9ac67686-c504-4bae-a908-87ced0458403',
 'cd44cd1f-fb72-4ce2-8aaa-fa743948b358',
 'b4b4faf7-3720-4bf1-8a17-3bdee7c450c9',
 'd5515a57-a417-4b12-9297-9a494d93034b',
 'f64d90b6-862c-46f0-aaad-c2052667762b',
 '3bd8cc88-b038-442f-bedf-bf41a64d971b',
 'a25dbd8d-422d-4c61-a6d7-7f668813b572',
 'f276b050-5946-45aa-92e9-729d9f39d999']

In [ ]:
#initialize the chatmodel
chatmodel = ChatOpenAI(
    api_key=api_key,
    base_url=base_url,
    model=chat_model_name,
    temperature=0
)

### The Retriever
Retrieving Documents for a Query

In [ ]:
# Maximal Marginal Relevance (MMR) for diverse and relevant results.
msme_retriever = msmevdb.as_retriever(search_type="mmr")

k defines the final number of documents that the retriever will return. use for concise and relevant results instead of retrieving too much data.

fetch_k determines how many documents are initially retrieved before filtering.

In [ ]:
msme_retriever = msmevdb.as_retriever(search_type="mmr", search_kwargs={'k': 4, 'fetch_k': 10})

Retrieves related documents based on `question`

In [ ]:
question = "How long does a business name reservation last in Nigeria?"
msme_retriever.invoke(question)

**Generating Responses Using Llama 3.3 70b  model**

- Let us re-create the chat model for response generation.
- **`temperature= 0` ensures deterministic outputs.**
- You can point `model` at any chat model your provider offers - just change the
  name below, or change `chat_model_name` in the setup cell to change it everywhere.

In [ ]:
#initialize the chatmodel
chatmodel = ChatOpenAI(
    api_key=api_key,
    base_url=base_url,
    model=chat_model_name,
    temperature=0
)

### **Prompting the LLM**

Creates a prompt template (prompt) with a system prompt and placeholders for the context and user question

In [ ]:
#using from template method
prompt = ChatPromptTemplate.from_template(
    """You are a business consultant providing insights on MSMEs in Nigeria.
You will be provided with the context: {context} to answer the user's question.
The context includes sections on understanding, starting, growing, and sustaining MSMEs, policies, and industry-specific information.
Provide a comprehensive response.
Include relevant sources or links from the context in your response at the end of each answer, include a statement: "To read more, check out this link: [insert link]."
Avoid unnecessary or unrelated details. Format the text output clearly and professionally in an HTML format.
question: {question}""")




Initializes the chat completion chat model (llm) to use for generating responses

Creates a simple output parser (parse_output) to parse the LLM output as a string

Chains all the above components using LangChain’s pipe ( | ) notation to create a simple RAG workflow (rag_chain)

In [ ]:
question = "How long does a business name reservation last in Nigeria?"

#retrieve the document
get_doc = msme_retriever.invoke(question)
# Prepare the input for the chain
input = {"context": get_doc, "question": question}

# Create the chain
chain = prompt | chatmodel | StrOutputParser()

answer = chain.invoke(input)
print(answer)

**Using from messages method**

- Separates **system instructions and user input** into structured messages.
- Can be adapted for other structured conversations.


In [ ]:
# Define the prompt using from_messages
prompt_messages = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are a business consultant providing insights on MSMEs in Nigeria.
            You will be provided with the context: {context} to answer the user's question.
            The context includes sections on understanding, starting, growing, and sustaining MSMEs, policies, and industry-specific information.
            Provide a comprehensive response.
            Include relevant sources or links from the context in your response at the end of each answer, include a statement: "To read more, check out this link: [insert link]."
            Avoid unnecessary or unrelated details. Format the text output clearly and professionally in an HTML format."""
        ),
        ("human", "question: {question}"),
    ]
)

# Prepare the input for the chain
input = {"context": get_doc, "question": question}
# Create the chain
chain = prompt_messages | chatmodel | StrOutputParser()
# Generate and print the answer
result = chain.invoke(input)
print(result)

## **Other Retriever Techniques**

## Parent Document Retriever 


When preparing documents for LLMs:

Small chunks improve retrieval quality.
Large chunks maintain context for better generation.
Simple strategies like fixed or recursive splitting can't balance both. 

**Parent document retrieval solves this by:**
Embedding small chunks for retrieval.
Fetching larger chunks or source documents for context.

This ensures the LLM gets complete context, improving response quality. Useful for tasks needing expanded context.

#### import the necessary libraries 

We will require the following libraries for this method:

In [ ]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.schema import Document

We will be using the Parent Document Retriever library from LangChain.

For this retriever we do **not** want the small chunks we made earlier. We want
the **full pages** to act as the parent documents, and we let the retriever
create the small child chunks itself.

So we simply reload the file we want to use.

** This document consist of two attributes-page_content and metadata.**

In [ ]:
# Reload the sources, but this time keep the documents whole.
# These full pages become the "parent" documents.
pdf_docs = PyPDFLoader("msme_policy.pdf").load()

with open("business_registration_guide.txt", "r", encoding="utf-8") as f:
    txt_text = f.read()

# Wrap the text file into a single document, so both sources match
txt_docs = [Document(page_content=txt_text)]

# Choose the source, the same way we did earlier
chosen_docs = pdf_docs + txt_docs       # use both together
# chosen_docs = pdf_docs                # use only the PDF
# chosen_docs = txt_docs                # use only the text file

msme_documents = [Document(page_content=doc.page_content) for doc in chosen_docs]

print(f"Total Documents to be embedded: {len(msme_documents)}\n")

In [ ]:
#This is to divides the documents into smaller chunks
split_msme = RecursiveCharacterTextSplitter(chunk_size=500)

Now, let's create a vector store for storing smaller chunks and an InMemoryStore to store the parent documents.

The InMemoryStore functions as a key-value pair data structure, where:

Each key is a unique UUID assigned to a parent document.

Each value is the actual text content of the corresponding parent document.

This structure ensures that while the program is running, the parent documents remain accessible in memory, allowing efficient retrieval and reconstruction of full documents when needed.


In [ ]:
msme_vectorestore = Chroma(collection_name="msme_documents", embedding_function=embeddings)
# This store will retain parent documents for retrieval
store = InMemoryStore()

Initialize the Parent Document Retriever

In [ ]:
retriever = ParentDocumentRetriever(
    vectorstore = msme_vectorestore , 
    docstore=store, 
    child_splitter=split_msme
)

Add Documents to Retriever

In [ ]:
retriever.add_documents(msme_documents)

After adding, we can check how many keys are in the store. Each key is one
**parent** document - that is, one full PDF page, or the whole text file.

The number below should match the number of documents we printed earlier.

In [ ]:
len(list(store.yield_keys()))

 This will only returns the small chunks

 Search for relevant document chunks using the vector store

In [ ]:
sub_docs = msme_vectorestore.similarity_search("What is the first step in setting up a business in Nigeria?")
sub_docs

: Retrieve Relevant Parent Documents

In [ ]:
retrieved_docs = retriever.get_relevant_documents("What is the first step in setting up a business in Nigeria?")

print(retrieved_docs[0].page_content)

### Generate Response 

we will use the same prompt above

In [ ]:
question = "How long does a business name reservation last in Nigeria?"

#retrieve the document
retrieved_docs = retriever.get_relevant_documents(question)

# Prepare the input for the chain
input = {"context": retrieved_docs, "question": question}

# Create the chain
chain = prompt | chatmodel | StrOutputParser()

answer = chain.invoke(input)
print(answer)

### Multiple Query Retriever

This method enhances document retrieval by generating multiple variations of a user query, increasing the likelihood of retrieving relevant documents from a vector database

 We will prompt the LLM to create five different variations of the user’s query.
Helps overcome limitations of distance-based similarity search, ensuring that relevant documents aren't missed due to minor phrasing differences.

In [ ]:

template = """You are an AI language model assistant. Your task is to generate five 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple type of the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. Do not add explanation or numbers to the question, only 
generate the questions.
Provide these alternative questions separated by newlines Original question: {question}"""
multiple_query_prompt = ChatPromptTemplate.from_template(template)

Alternative versions of the query, which are split into individual questions using split("\n") to create a list.

Since the LLM generates multiple queries as a block of text with line breaks, we split the text into individual lines to get distinct query variations.

Using .strip() prevents storing empty or whitespace-only strings in the final output.

In [ ]:
# question = "How long does a business name reservation last in Nigeria?"
question ="what is the repayment plan for loan from Development Bank of Nigeria"

multiple_queries = multiple_query_prompt | chatmodel | StrOutputParser() | (lambda x: [q for q in x.split("\n") if q.strip()])
multiple_answer = multiple_queries.invoke({"question" : question})
print(multiple_answer)

Define the Retriever

In [ ]:
msme_retriever = msmevdb.as_retriever()

Get Unique Documents


Let us Convert the documents into JSON strings using dumps() and also remove duplicates using (set), and convert them back to objects using loads().

This is to ensure the retrieved documents are unique.

In [ ]:
from langchain.load import dumps, loads
def unique_doc (documents): 
    unique_doc = list(set(dumps(doc)for doc in documents))
    return [loads(doc) for doc in unique_doc]

Retrieving Documents Using Multiple Queries

With .map(), The msme_retriever processes each query separately, retrieving relevant documents for each one instead of handling all queries at once.

In [ ]:
retrieval_chain = multiple_queries | msme_retriever.map() | unique_doc

Answer Generation Using Retrieved Documents

 We pass all the retrieved documents as context along with the original question to the llm

In [ ]:

prompt = ChatPromptTemplate.from_template(
    """You are a business consultant providing insights on MSMEs in Nigeria.
You will be provided with the context: {context} to answer the user's question.
The context includes sections on understanding, starting, growing, and sustaining MSMEs, policies, and industry-specific information.
Provide a detailed and comprehensive response.
Include relevant sources or links from the context in your response. At the end of each answer, include a statement: "To read more, check out this link: [insert link]."
Avoid unnecessary or unrelated details. Format the text output clearly and professionally in an HTML format in not more than 5 sentences.
question: {question}""")

multiple_query_input = ({"context" : retrieval_chain, "question": question})

chain = prompt| chatmodel| StrOutputParser()

answer = chain.invoke(multiple_query_input)
print(answer)  

### BM25 (BEST MATCHING 25)

**Problem:** Dense models sometimes overlook exact keywords in queries.  
**Solution:** Uses term frequency (TF-IDF) to rank documents with strong keyword overlap.

- Great for domains with specific terms (e.g., law, code)
- Useful for hybrid retrieval (dense + sparse)
- Fast and interpretable baseline

 Imports and Setup the neccessary libraries

In [ ]:
from langchain.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain.schema import Document
from nltk.tokenize import word_tokenize
import nltk

# Downloads tokenization rules
nltk.download("punkt_tab") 


**Load the Vector Store (Semantic Retriever)**

In [ ]:
msmevdb = Chroma(
    collection_name="msme",
    embedding_function=embeddings,
    persist_directory=chroma_path
)

** Extract Raw Documents and Convert to LangChain Format**

You're fetching all the stored text back out of Chroma.

Then wrapping them as LangChain Document objects so they can be passed to other retrievers (like BM25).

In [ ]:
question_docs = msmevdb.get(include=['documents'])
question_documents = [Document(page_content=doc) for doc in question_docs['documents']]

print("Documents pulled back out of Chroma:", len(question_documents))

**Create Two Individual Retrievers**

In [ ]:
question_semantic = msmevdb.as_retriever()
question_bm25 = BM25Retriever.from_documents(question_documents, preprocess_func=word_tokenize)

**Combine Both with EnsembleRetriever**

Combines both retrievers.

The weights=[0.5, 0.5] means equal influence from semantic and keyword-based scores.

You can adjust the weights to favor one over the other.

In [ ]:
bm25_retriever = EnsembleRetriever(retrievers=[question_semantic, question_bm25], weights=[0.5, 0.5])

In [ ]:
question = "How do i register a construction business Nigeria"
results = bm25_retriever.get_relevant_documents(question )
results

In [ ]:
# Prepare the input for the chain
input = {"context": results , "question": question}

# Create the chain
chain = prompt | chatmodel | StrOutputParser()

answer = chain.invoke(input)
print(answer)

### RAG EVALUATION TECHNIQUES

#### RAG Evaluation using RAGAS

Retrieval-Augmented Generation (RAG) systems should not just “sound correct” — they must retrieve accurate, relevant, and faithful content.

We use RAGAS (Retrieval-Augmented Generation Assessment Suite) to measure how well the system performs on both retrieval and generation steps.

In [ ]:
from langchain_openai import ChatOpenAI

**Sample Queries and References (Ground Truth)**

These represent typical user questions and what the system should ideally answer.

They form the evaluation dataset.

In [ ]:
sample_queries = [
    "What are the classification criteria for MSMEs in Nigeria according to the National Policy on MSMEs?",
    "How do SMEDAN, BoI, and CBN define micro, small, and medium enterprises differently?",
    "What are the key challenges faced by MSMEs in Nigeria?",
    "Which sectors dominate micro enterprises in Nigeria, and what percentage do they represent?",
    "What government interventions or policies have been introduced to support MSMEs in Nigeria?"
]

expected_responses = [
    "The National Policy on MSMEs classifies enterprises as: Micro (<10 employees, assets <₦10M), Small (10-49 employees, assets ₦10M-₦100M), and Medium (50-199 employees, assets ₦100M-₦1B). Employment criteria take precedence if conflicts arise with asset thresholds.",
    "SMEDAN defines Micro (<10 employees, <₦5M assets), Small (10-49 employees, ₦5M-₦50M assets), and Medium (50-199 employees, ₦50M-₦500M assets). BoI and CBN use similar employee ranges but differ slightly in turnover/asset thresholds (e.g., BoI includes turnover <₦20M for Micro).",
    "Key challenges include poor access to credit, low market access, weak infrastructure, discriminatory legislation, and lack of technical skills. Micro enterprises also face funding constraints (reliance on personal savings/community funds).",
    "Micro enterprises are dominated by wholesale/retail trade (54.67%), followed by manufacturing (13.21%), agriculture (8.92%), and services (7.80%). The 2013 survey estimated 36.99 million micro enterprises in Nigeria.",
    "Interventions include SMEDAN (capacity building), BOI/CBN loan schemes, the Finance Act 2019/2020 (tax exemptions for small businesses), and programs like the National Economic Reconstruction Fund. Historical schemes date back to the 1970s (e.g., Agricultural Credit Guarantee Scheme)."
]

**Evaluate a Specific Retriever (e.g. BM25)**



In [ ]:
#Let us evaluate one of our retriever. lets evaluate the BM25

relevant_docs=bm25_retriever.get_relevant_documents(question )

#### Run the Generation Chain

This chain creates the final LLM answer based on the retrieved context and the query.

In [ ]:
# Prepare the input for the chain
input = {"context":  relevant_docs , "question": question}

# Create the chain
chain = prompt | chatmodel | StrOutputParser()


#### Build the Evaluation Dataset

For each query:

You retrieve documents

Run the LLM chain

Store the question, context, model answer, and reference answer in a list

In [ ]:
dataset = []

for query, reference in zip(sample_queries, expected_responses):
    # Get relevant documents
    relevant_docs = bm25_retriever.get_relevant_documents(query)
    
    # Convert Document objects to strings for the context
    retrieved_contexts = [doc.page_content for doc in relevant_docs]
    
    # Get the model's response
    response = chain.invoke({"question": query, "context": retrieved_contexts})  # Adjust if your chain expects different input
    
    dataset.append({
        "user_input": query,        # Required by context_recall
        "contexts": retrieved_contexts,  # Used by context precision/relevancy
        "retrieved_contexts": retrieved_contexts,  # Required by context_recall
        "response": response,         # The generated answer
        "reference": reference     # Required by context_recall
    })


**Now, load the dataset into EvaluationDataset object.**

In [ ]:
from ragas import EvaluationDataset

evaluation_dataset = EvaluationDataset.from_list(dataset)

#### Set Up the LLM Evaluator


In [ ]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper

llm = ChatOpenAI(
    api_key=api_key,
    base_url=base_url,
    model=chat_model_name,
    temperature=0
)
evaluator_llm = LangchainLLMWrapper(llm)

In [ ]:
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness,  ResponseRelevancy

#### Choose RAGAS Metrics

LLMContextRecall: Measures if enough relevant info was retrieved

Faithfulness: Checks if the answer sticks to the retrieved info (no hallucinations)

FactualCorrectness: Verifies if the answer is factually accurate compared to the

##### Run Evaluation

In [ ]:
result = evaluate(dataset=evaluation_dataset,metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness()],llm=evaluator_llm)
result

**Result of our evaluation metrics**

Context Recall (0.90): The retriever is doing very well — it returns most of the relevant documents.

Faithfulness (0.98): The LLM sticks closely to the retrieved context — very low hallucination.

 Factual Correctness (0.59): Many answers miss details or don’t fully match the reference — accuracy needs improvement.